In [30]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [46]:
parsnp_data = pd.read_csv('/home/Users/rdd4/bronko-test/bronko_test/alignment_comparisons/hiv_evolution_results/S2/parsnp_100_consensus/parsnp.vcf', delimiter='\t', skiprows=5)
ska_data = pd.read_csv('/home/Users/rdd4/bronko-test/bronko_test/alignment_comparisons/hiv_evolution_results/S2/ska/alignment.vcf', delimiter='\t', skiprows=2)

parsnp_data = parsnp_data.drop(columns=['ancestral_sequence.fa.ref'])

## add .fasta to all cols in ska_data, remove . instances or replace as 0
ska_data.columns = (
    list(ska_data.columns[:9]) +
    [f"{col}.fasta" for col in ska_data.columns[9:]]
)

ska_data = ska_data[ska_data['ALT'] != '.']
ska_data = ska_data.replace('.', 0)

/tmp/ipykernel_3353707/1942356965.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ska_data = ska_data.replace('.', 0)


In [47]:
k = 21  # window size

pos = parsnp_data["POS"].to_numpy()
sample_cols = parsnp_data.columns[9:]

results = {}
close_locations = {}  # store tuples of nearby variant positions

num_variants_per_sample = []

for col in sample_cols:
    idx = np.where(parsnp_data[col].to_numpy() == 1)[0]
    num_variants_per_sample.append(len(idx))
    
    if len(idx) < 2:
        results[col] = 0
        close_locations[col] = []
        continue
    
    sample_pos = pos[idx]
    diffs = np.diff(sample_pos)
    
    threshold = k//2 + 1
    close_mask = diffs <= threshold
    
    # ---- collect tuples ----
    pairs = [
        (sample_pos[i], sample_pos[i+1])
        for i in np.where(close_mask)[0]
    ]
    close_locations[col] = pairs
    
    # ---- counting logic (your original method) ----
    count = (
        close_mask.sum() * 2
        - np.sum(close_mask[:-1] & close_mask[1:])
    )
    
    results[col] = count

result_series = pd.Series(results)

result_series
result_series.sum()

np.int64(983)

In [48]:
print(max(num_variants_per_sample))
print(min(num_variants_per_sample))
print(np.median(num_variants_per_sample))

134
57
101.0


In [49]:
k = 21  # window size

pos = parsnp_data["POS"].to_numpy()
sample_cols = parsnp_data.columns[9:]

results = {}
cluster_locations = {}

for col in sample_cols:
    idx = np.where(parsnp_data[col].to_numpy() == 1)[0]
    
    if len(idx) < 3:
        results[col] = 0
        cluster_locations[col] = []
        continue
    
    sample_pos = pos[idx]
    
    clusters = []
    counted = set()
    
    left = 0
    for right in range(len(sample_pos)):
        
        # shrink window if too large
        while sample_pos[right] - sample_pos[left] > k:
            left += 1
        
        window_size = right - left + 1
        
        if window_size >= 3:
            cluster = tuple(sample_pos[left:right+1])
            clusters.append(cluster)
            counted.update(cluster)
    
    cluster_locations[col] = clusters
    results[col] = len(counted)

result_series = pd.Series(results)

result_series
result_series.sum()

np.int64(286)

In [50]:
ska_data

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,0-42-0.284972.fasta,...,62-48-0.296761.fasta,64-8-0.029273.fasta,6-57-0.351311.fasta,66-89-0.102048.fasta,68-27-0.141118.fasta,70-88-0.403171.fasta,72-98-0.242475.fasta,74-3-0.458640.fasta,76-17-0.002690.fasta,8-4-0.233787.fasta
19,Ancestral,35,0,G,"C,A",0,0,0,GT,0,...,0,0,1,2,2,2,2,0,0,1
20,Ancestral,42,0,G,A,0,0,0,GT,0,...,1,0,0,0,0,0,0,0,0,0
21,Ancestral,69,0,T,C,0,0,0,GT,0,...,0,0,0,0,0,0,1,0,0,0
22,Ancestral,86,0,A,T,0,0,0,GT,0,...,1,1,0,1,1,1,1,1,1,0
23,Ancestral,89,0,G,A,0,0,0,GT,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2473,Ancestral,9132,0,C,T,0,0,0,GT,0,...,0,0,0,0,0,0,0,0,0,0
2474,Ancestral,9133,0,A,G,0,0,0,GT,0,...,0,0,1,0,0,0,0,0,0,1
2475,Ancestral,9171,0,A,G,0,0,0,GT,0,...,0,0,0,0,0,0,0,0,0,0
2476,Ancestral,9172,0,G,A,0,0,0,GT,0,...,0,0,0,0,0,0,0,0,0,0


In [51]:
parsnp_data

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,50-83-0.062133.fasta,...,32-99-0.358423.fasta,40-34-0.239630.fasta,18-82-0.208535.fasta,2-16-0.175737.fasta,22-76-0.026383.fasta,34-73-0.018221.fasta,36-93-0.030735.fasta,4-20-0.054944.fasta,10-90-0.039065.fasta,20-86-0.311521.fasta
0,Ancestral,35,ACCAAGCACC.GCTAATCGGA,G,"A,C",40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
1,Ancestral,42,ACCGCTAATC.GGATAGAAAT,G,A,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
2,Ancestral,69,TAGCAAAATG.TCGGAACAAT,T,C,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
3,Ancestral,86,AATTATAGAA.AATGACTTCA,A,T,40,PASS,NaN,GT,1,...,0,1,0,0,0,0,1,0,0,0
4,Ancestral,89,TATAGAAAAT.GACTTCAGGA,G,A,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
897,Ancestral,9132,GCAAGTATTA.CAGATGTCAG,C,T,40,PASS,NaN,GT,0,...,1,0,0,0,0,1,0,0,0,0
898,Ancestral,9133,CAAGTATTAC.AGATGTCAGG,A,G,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
899,Ancestral,9171,GTACCATGGC.AGATAACTAT,A,G,40,PASS,NaN,GT,0,...,0,0,0,0,0,0,0,0,0,0
900,Ancestral,9172,TACCATGGCA.GATAACTATT,G,A,40,PASS,NaN,GT,0,...,0,1,0,0,0,0,1,0,0,0


In [52]:
len([col for col in parsnp_data.columns if col.endswith(".fasta")])

39

In [53]:
len([col for col in ska_data.columns if col.endswith(".fasta")])

39

In [54]:
def split_multiple_alt_alleles(df):

    df = df.copy()
    fasta_cols = df.columns[9:]  # sample columns

    multi_mask = df["ALT"].str.contains(",", na=False)

    # Separate multi-ALT rows
    df_multi = df[multi_mask].copy()
    df_single = df[~multi_mask].copy()

    new_rows = []

    for _, row in df_multi.iterrows():

        alts = row["ALT"].split(",")

        for i, alt in enumerate(alts, start=1):  # ALT index = 1-based

            new_row = row.copy()
            new_row["ALT"] = alt

            # Convert sample columns:
            # keep only values equal to this alt index
            for col in fasta_cols:
                new_row[col] = 1 if row[col] == i else 0

            new_rows.append(new_row)

    df_multi_split = pd.DataFrame(new_rows)

    # Combine back
    result = pd.concat([df_single, df_multi_split], ignore_index=True)

    return result

In [55]:
parsnp_data['POS'] = parsnp_data['POS'].astype(int)
ska_data['POS'] = ska_data['POS'].astype(int)

parsnp_data = split_multiple_alt_alleles(parsnp_data)
ska_data = split_multiple_alt_alleles(ska_data)

In [62]:
import os
import pandas as pd


def get_bronko_df(location):
    vcf_files = [f for f in os.listdir(location) if f.endswith(".vcf")]

    variant_dict = {}

    for vcf_file in vcf_files:
        sample_name = vcf_file.replace(".vcf", ".fasta")
        with open(os.path.join(location, vcf_file)) as f:
            for line in f:
                if line.startswith("#"):
                    continue
                fields = line.strip().split("\t")
                chrom, pos, _, ref, alt = fields[:5]
                key = (chrom, pos, ref, alt)
                if key not in variant_dict:
                    variant_dict[key] = {}
                variant_dict[key][sample_name] = 1
                
    print(variant_dict)

    # convert to DataFrame

    bronko = pd.DataFrame.from_dict(variant_dict, orient="index")
    bronko = bronko.fillna(0).astype(int)
    bronko.reset_index(inplace=True)
    bronko.rename(columns={"index": ["CHROM", "POS", "REF", "ALT"]}, inplace=True)

    new_names = ['#CHROM','POS','REF','ALT']
    bronko.columns = new_names + bronko.columns[4:].tolist()
    bronko['POS'] = bronko['POS'].astype(int)
    return bronko

bronko = get_bronko_df("/home/Users/rdd4/bronko-test/bronko_test/alignment_comparisons/hiv_evolution_results/S2/bronko_single_ref")  # change this to your VCF folder path
bronko

# df = bronko.drop(columns='index')

# # save table

# # df.to_csv("variant_presence_matrix.csv", index=False)
# print("Saved variant presence matrix to variant_presence_matrix.csv")


{('Ancestral', '35', 'G', 'C'): {'6-57-0.351311.fasta': 1, '8-4-0.233787.fasta': 1}, ('Ancestral', '122', 'G', 'A'): {'6-57-0.351311.fasta': 1, '8-4-0.233787.fasta': 1, '58-68-0.341532.fasta': 1}, ('Ancestral', '217', 'C', 'A'): {'6-57-0.351311.fasta': 1, '8-4-0.233787.fasta': 1}, ('Ancestral', '241', 'G', 'A'): {'6-57-0.351311.fasta': 1, '8-4-0.233787.fasta': 1}, ('Ancestral', '248', 'A', 'G'): {'6-57-0.351311.fasta': 1, '8-4-0.233787.fasta': 1, '2-16-0.175737.fasta': 1, '4-20-0.054944.fasta': 1}, ('Ancestral', '346', 'A', 'G'): {'6-57-0.351311.fasta': 1, '8-4-0.233787.fasta': 1, '2-16-0.175737.fasta': 1, '4-20-0.054944.fasta': 1}, ('Ancestral', '468', 'G', 'A'): {'6-57-0.351311.fasta': 1, '52-91-0.290508.fasta': 1, '60-36-0.213448.fasta': 1, '72-98-0.242475.fasta': 1, '44-31-0.159809.fasta': 1, '8-4-0.233787.fasta': 1, '74-3-0.458640.fasta': 1, '46-43-0.202098.fasta': 1, '58-68-0.341532.fasta': 1, '40-34-0.239630.fasta': 1, '62-48-0.296761.fasta': 1, '64-8-0.029273.fasta': 1, '36-93-

,#CHROM,POS,REF,ALT,6-57-0.351311.fasta,8-4-0.233787.fasta,58-68-0.341532.fasta,2-16-0.175737.fasta,4-20-0.054944.fasta,52-91-0.290508.fasta,...,16-11-0.108007.fasta,20-86-0.311521.fasta,34-73-0.018221.fasta,18-82-0.208535.fasta,30-70-0.261161.fasta,0-42-0.284972.fasta,26-55-0.451799.fasta,12-66-0.270719.fasta,24-41-0.253877.fasta,22-76-0.026383.fasta
0,Ancestral,35,G,C,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Ancestral,122,G,A,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Ancestral,217,C,A,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Ancestral,241,G,A,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Ancestral,248,A,G,1,1,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
928,Ancestral,6421,C,T,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
929,Ancestral,893,A,G,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
930,Ancestral,1538,T,C,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
931,Ancestral,4174,A,G,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1


In [63]:
def get_shared(parsnp_data, ska_data, bronko):
    df1 = parsnp_data
    df2 = ska_data
    df3 = bronko

    variant_cols = ["#CHROM", "POS", "REF", "ALT"]

    def add_suffix(df, suffix):
        sample_cols = [c for c in df.columns if c not in variant_cols]
        return df.rename(columns={c: f"{c}_{suffix}" for c in sample_cols})

    df1_ = add_suffix(df1, "1")
    df2_ = add_suffix(df2, "2")
    df3_ = add_suffix(df3, "3")

    merged = (
        df1_
        .merge(df2_, on=variant_cols, how="outer")
        .merge(df3_, on=variant_cols, how="outer")
    )
    

    fasta_cols = [c for c in merged.columns if ".fasta" in c]

    merged[fasta_cols] = (
        merged[fasta_cols]
        .fillna(0)
        .astype(int)
    )

    sample_cols = [c for c in df1.columns[9:] if c not in variant_cols]


    
    results = []
    ska_missed = []

    for c in sample_cols:
        col1 = f"{c}_1"
        col2 = f"{c}_2"
        col3 = f"{c}_3"

        a = merged[col1] == 1
        b = merged[col2] == 1
        d = merged[col3] == 1

        only_1 = (a & ~b & ~d).sum()
        only_2 = (~a & b & ~d).sum()
        only_3 = (~a & ~b & d).sum()

        shared_12_only = (a & b & ~d).sum()
        shared_13_only = (a & ~b & d).sum()
        shared_23_only = (~a & b & d).sum()

        shared_123 = (a & b & d).sum()
        
        if shared_13_only:
            print(c)
            ska_missed.append(c)

        results.append({
            "sample": c,
            "only_1": int(only_1),
            "only_2": int(only_2),
            "only_3": int(only_3),
            "shared_12_only": int(shared_12_only),
            "shared_13_only": int(shared_13_only),
            "shared_23_only": int(shared_23_only),
            "shared_123": int(shared_123)
        })

    overlap_df = pd.DataFrame(results)
    overlap_df
    return overlap_df, merged, ska_missed

In [64]:
overlap_df, merged, ska_missed = get_shared(parsnp_data, ska_data, bronko)
overlap_df

50-83-0.062133.fasta
68-27-0.141118.fasta
14-56-0.046590.fasta
28-30-0.432022.fasta
26-55-0.451799.fasta
72-98-0.242475.fasta
6-57-0.351311.fasta
62-48-0.296761.fasta
64-8-0.029273.fasta
48-50-0.031841.fasta
56-49-0.256628.fasta
46-43-0.202098.fasta
30-70-0.261161.fasta
16-11-0.108007.fasta
24-41-0.253877.fasta
70-88-0.403171.fasta
52-91-0.290508.fasta
60-36-0.213448.fasta
44-31-0.159809.fasta
54-96-0.220146.fasta
74-3-0.458640.fasta
58-68-0.341532.fasta
42-59-0.104116.fasta
38-37-0.022718.fasta
8-4-0.233787.fasta
66-89-0.102048.fasta
12-66-0.270719.fasta
0-42-0.284972.fasta
76-17-0.002690.fasta
32-99-0.358423.fasta
40-34-0.239630.fasta
18-82-0.208535.fasta
2-16-0.175737.fasta
22-76-0.026383.fasta
34-73-0.018221.fasta
36-93-0.030735.fasta
4-20-0.054944.fasta
10-90-0.039065.fasta
20-86-0.311521.fasta


,sample,only_1,only_2,only_3,shared_12_only,shared_13_only,shared_23_only,shared_123
0,50-83-0.062133.fasta,4,0,0,0,9,0,53
1,68-27-0.141118.fasta,6,0,0,0,41,0,79
2,14-56-0.046590.fasta,3,0,0,0,30,0,75
3,28-30-0.432022.fasta,5,0,0,0,29,0,50
4,26-55-0.451799.fasta,5,0,0,0,49,0,71
5,72-98-0.242475.fasta,4,0,0,0,40,0,77
6,6-57-0.351311.fasta,3,0,0,0,37,0,69
7,62-48-0.296761.fasta,6,0,0,0,29,0,66
8,64-8-0.029273.fasta,1,0,0,0,21,0,43
9,48-50-0.031841.fasta,4,0,0,0,14,0,58


In [65]:
overlap_df.iloc[:, 1:].sum()

only_1             134
only_2               0
only_3               0
shared_12_only       0
shared_13_only    1231
shared_23_only       0
shared_123        2523
dtype: int64

In [66]:
(2523+1231)/3822

0.9822082679225537

In [61]:
overlap_df.iloc[:, 1:].sum().sum()

np.int64(3888)

In [19]:
results = {}

for sample in ska_missed:
    cols = ["POS", "REF", "ALT"] + [f"{sample}_{i}" for i in [1,2,3]]
    data = merged[merged[f"{sample}_1"] == 1][cols]
    
    data = data[data[[f"{sample}_{i}" for i in [1,2,3]]].sum(axis=1) != 3]
    
    data = data.rename(columns={
        f"{sample}_1": "parsnp",
        f"{sample}_2": "ska",
        f"{sample}_3": "bronko"
    })
    
    results[sample] = data
    
final_df = (
    pd.concat(results, names=["sample"])
      .reset_index(level=0)
      .reset_index(drop=True)
)
final_df

,sample,POS,REF,ALT,parsnp,ska,bronko
0,68-27-0.141118.fasta,35,G,A,1,0,1
1,68-27-0.141118.fasta,462,C,T,1,0,1
2,68-27-0.141118.fasta,468,G,A,1,0,1
3,68-27-0.141118.fasta,471,A,G,1,0,0
4,68-27-0.141118.fasta,654,C,A,1,0,1
...,...,...,...,...,...,...,...
1347,20-86-0.311521.fasta,6922,A,G,1,0,1
1348,20-86-0.311521.fasta,6931,G,A,1,0,1
1349,20-86-0.311521.fasta,7802,G,A,1,0,1
1350,20-86-0.311521.fasta,7807,G,A,1,0,1


In [ ]:
ska_data[ska_data['POS']==1566][['POS', 'REF', 'ALT', '68-27-0.141118']]

,POS,REF,ALT,24-68-0.272818.fasta
46,1566,G,A,1
